# Practice Session
Riders set lap times in bike_number order. Best lap determines final classification.

In [ ]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

sys.path.insert(0, '..')
from src.engine import circuit_weights, perf_score, simulate_lap, fmt_lap, fmt_gap
from src.loader import load_riders, load_circuits

RAW  = Path('../data/raw')
LAPS = 15

In [12]:
entry_info    = pd.read_csv(RAW / 'entry_info.csv')
riders_rating = pd.read_csv(RAW / 'riders_rating.csv').rename(columns={
    'braking': 'rider_braking', 'cornering': 'rider_cornering'
})
bikes_rating  = pd.read_csv(RAW / 'bikes_rating.csv').rename(columns={
    'braking': 'bike_braking', 'cornering': 'bike_cornering'
})
circuits = pd.read_csv(RAW / 'circuits.csv')

df = (
    entry_info
    .merge(riders_rating, on='name', how='left')
    .merge(bikes_rating,  on=['manufacturer', 'team_status'], how='left')
)

print(f'Riders loaded : {len(df)}')
print(f'Circuits loaded: {len(circuits)}')

Riders loaded : 24
Circuits loaded: 13


In [13]:
# Browse available circuits
display(
    circuits[['circuit_name', 'country', 'lap_length_km', 'base_lap_time', 'corners', 'straight_length_m']]
    .reset_index()
    .rename(columns={'index': 'idx'})
)

,idx,circuit_name,country,lap_length_km,base_lap_time,corners,straight_length_m
0,0,Liam Henderson Circuit,Australia,4.4,87,11,1050
1,1,Sagiwara Grand Prix Circuit,Japan,4.9,105,14,920
2,2,Autodromo di Valdivento,Italy,5.1,105,15,980
3,3,Circuito de Las Dunas,Spain,4.6,98,13,850
4,4,Schwarzwald Arena,Germany,3.8,82,14,650
5,5,Northmoor International Circuit,United Kingdom,5.7,128,17,1100
6,6,Autódromo do Cerrado,Brazil,4.8,100,14,900
7,7,Autódromo de la Pampa,Argentina,4.7,96,13,870
8,8,Tulpenring,Netherlands,4.5,91,18,700
9,9,Circuit de la Vallée du Rhône,France,4.2,91,13,780


In [14]:
# Select circuit by index (see table above)
CIRCUIT_IDX = 0
circuit = circuits.iloc[CIRCUIT_IDX]

print(f"Circuit : {circuit['circuit_name']}")
print(f"Country : {circuit['country']}")
print(f"Length  : {circuit['lap_length_km']} km  |  Base lap: {circuit['base_lap_time']}s  |  Corners: {circuit['corners']}  |  Straight: {circuit['straight_length_m']} m")

Circuit : Liam Henderson Circuit
Country : Australia
Length  : 4.4 km  |  Base lap: 87s  |  Corners: 11  |  Straight: 1050 m


In [16]:
# ── Practice Session ────────────────────────────────────────────────────────

w_spd, w_cor, w_brk = circuit_weights(circuit)
base_time = circuit['base_lap_time']

riders_sorted = df.sort_values('bike_number').reset_index(drop=True)

print('=' * 65)
print(f"  PRACTICE SESSION")
print(f"  {circuit['circuit_name'].upper()}  —  {circuit['country'].upper()}")
print(f"  {circuit['lap_length_km']} km | {circuit['corners']} corners | {circuit['straight_length_m']} m straight")
print('=' * 65)

session_results = []

for _, rider in riders_sorted.iterrows():
    score     = perf_score(rider, w_spd, w_cor, w_brk)
    lap_times = []

    print(f"\n  #{rider['bike_number']:02d}  {rider['name']}  |  {rider['team']}  [{rider['team_status'].upper()}]")
    print(f"  {'LAP':>5}   {'TIME':>10}   {'BEST':>10}")
    print(f"  {'─'*32}")

    for lap_num in range(1, LAPS + 1):
        lap_sec  = simulate_lap(rider, base_time, lap_num, score)
        prev_best = min(lap_times) if lap_times else float('inf')
        lap_times.append(lap_sec)
        best = min(lap_times)

        pb_marker = '  ◄ PB' if best < prev_best else ''
        print(f"  Lap {lap_num:2d}   {fmt_lap(lap_sec):>10}   {fmt_lap(best):>10}{pb_marker}")

    session_results.append({
        'bike_number'  : int(rider['bike_number']),
        'name'         : rider['name'],
        'team'         : rider['team'],
        'team_status'  : rider['team_status'],
        'manufacturer' : rider['manufacturer'],
        'best_lap_sec' : min(lap_times),
        'best_lap'     : fmt_lap(min(lap_times)),
    })

print('\n' + '=' * 65)
print('  SESSION COMPLETE')
print('=' * 65)

  PRACTICE SESSION
  LIAM HENDERSON CIRCUIT  —  AUSTRALIA
  4.4 km | 11 corners | 1050 m straight

  #07  Petros Georgiou  |  Honda Factory Racing  [FACTORY]
    LAP         TIME         BEST
  ────────────────────────────────
  Lap  1    01:29.705    01:29.705  ◄ PB
  Lap  2    01:29.503    01:29.503  ◄ PB
  Lap  3    01:29.223    01:29.223  ◄ PB
  Lap  4    01:28.711    01:28.711  ◄ PB
  Lap  5    01:28.422    01:28.422  ◄ PB
  Lap  6    01:28.302    01:28.302  ◄ PB
  Lap  7    01:28.338    01:28.302
  Lap  8    01:28.684    01:28.302
  Lap  9    01:28.609    01:28.302
  Lap 10    01:28.540    01:28.302
  Lap 11    01:28.578    01:28.302
  Lap 12    01:28.416    01:28.302
  Lap 13    01:28.424    01:28.302
  Lap 14    01:28.705    01:28.302
  Lap 15    01:28.497    01:28.302

  #10  Victor Burgos  |  Inferno Factory  [SATELLITE]
    LAP         TIME         BEST
  ────────────────────────────────
  Lap  1    01:30.060    01:30.060  ◄ PB
  Lap  2    01:29.842    01:29.842  ◄ PB
  Lap 

In [17]:
# ── Final Classification ─────────────────────────────────────────────────────

results_df = (
    pd.DataFrame(session_results)
    .sort_values('best_lap_sec')
    .reset_index(drop=True)
)
results_df['gap'] = results_df['best_lap_sec'] - results_df['best_lap_sec'].iloc[0]
results_df['gap_fmt'] = results_df['gap'].apply(fmt_gap)
results_df.index += 1

print('=' * 82)
print(f"  PRACTICE — FINAL CLASSIFICATION")
print(f"  {circuit['circuit_name']}  ({circuit['country']})")
print('=' * 82)
print(f"  {'P':<4} {'#':<5} {'RIDER':<24} {'TEAM':<26} {'MANUFACTURER':<14} {'BEST LAP':>10} {'GAP':>8}")
print(f"  {'─'*78}")

for pos, row in results_df.iterrows():
    print(f"  P{pos:<3} #{row['bike_number']:<4} {row['name']:<24} {row['team']:<26} {row['manufacturer']:<14} {row['best_lap']:>10} {row['gap_fmt']:>8}")

print('=' * 82)

  PRACTICE — FINAL CLASSIFICATION
  Liam Henderson Circuit  (Australia)
  P    #     RIDER                    TEAM                       MANUFACTURER     BEST LAP      GAP
  ──────────────────────────────────────────────────────────────────────────────
  P1   #91   Mehmet Terzioğlu         Ducati Factory Racing      Ducati          01:27.817        —
  P2   #31   Niklas Hoffmann          Suzuki Factory Racing      Suzuki          01:27.841   +0.023
  P3   #88   Sebastian Aginaza        Razor Racing               Ducati          01:27.866   +0.049
  P4   #44   Lorenzo Russo            Yamaha Factory Racing      Yamaha          01:27.942   +0.125
  P5   #20   Kaito Sato               Kawasaki Factory Racing    Kawasaki        01:28.083   +0.265
  P6   #90   Nathan Stewart           Ducati Factory Racing      Ducati          01:28.176   +0.359
  P7   #19   Matteo Esposito          Honda Factory Racing       Honda           01:28.267   +0.450
  P8   #32   Dwi Gunawan              Falcon Ra

In [18]:
# ── Export Practice Report ───────────────────────────────────────────────────

country     = circuit['country']
report_dir  = Path('../report') / country
report_dir.mkdir(parents=True, exist_ok=True)

title = f"{circuit['circuit_name']} - GRAND PRIX OF {country.upper()}"

# Markdown table
header = "| P | # | RIDER | TEAM | MANUFACTURER | BEST LAP | GAP |"
sep    = "|---|---|-------|------|--------------|----------|-----|"
rows   = []
for pos, row in results_df.iterrows():
    gap = row['gap_fmt']
    rows.append(
        f"| P{pos} | #{row['bike_number']} | {row['name']} | {row['team']} "
        f"| {row['manufacturer']} | {row['best_lap']} | {gap} |"
    )

md_content = f"# {title}\n\n## Practice - Final Classification\n\n{header}\n{sep}\n" + "\n".join(rows) + "\n"

out_path = report_dir / 'Practice.md'
out_path.write_text(md_content, encoding='utf-8')
print(f"Report saved: {out_path}")

Report saved: ..\report\Australia\Practice.md
